# Парсинг логов скрипта мониторинга

Парсим логи скрипта, мониторящего потребление памяти топ 10 (или скольки-то еще) процессов

In [11]:
import os
import sys
import pandas as pd
import numpy as np

In [38]:
envfilepath = None
if envfilepath is None:
    print("jopa")
    vardirpath = os.getcwd()
    
    #print(vardirpath)
    print(os.getcwd())
#     print(os.curdir())
    # print(dir(os))
    print(dir(os.path))
    print(os.path.dirname(vardirpath))

jopa
/Users/kot/Education/Projects/DevOps/tiny-monitor/analysis
['__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_get_sep', '_joinrealpath', '_varprog', '_varprogb', 'abspath', 'altsep', 'basename', 'commonpath', 'commonprefix', 'curdir', 'defpath', 'devnull', 'dirname', 'exists', 'expanduser', 'expandvars', 'extsep', 'genericpath', 'getatime', 'getctime', 'getmtime', 'getsize', 'isabs', 'isdir', 'isfile', 'islink', 'ismount', 'join', 'lexists', 'normcase', 'normpath', 'os', 'pardir', 'pathsep', 'realpath', 'relpath', 'samefile', 'sameopenfile', 'samestat', 'sep', 'split', 'splitdrive', 'splitext', 'stat', 'supports_unicode_filenames', 'sys']
/Users/kot/Education/Projects/DevOps/tiny-monitor


In [2]:
df = pd.read_csv('2024-02-16_tinymonitor_example.csv', sep='|')
display(df.head())
print(df.info())

,N,DATE,TIME,PID,RSS,VSZ,MEM,CPU,CMD,MAXMAPCOUNT,MAPS
0,1,2024-02-16,00:00:02,563127,1331,4491,16.7,0.1,java,65530,NaN
1,2,2024-02-16,00:00:02,3290,846,4490,10.6,0.1,java,65530,NaN
2,3,2024-02-16,00:00:02,18023,529,1717,6.6,0.1,java,65530,NaN
3,4,2024-02-16,00:00:02,675835,502,842,6.3,0.0,/usr/local/bin/python,65530,NaN
4,5,2024-02-16,00:00:02,675837,477,847,6.0,0.0,/usr/local/bin/python,65530,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14400 entries, 0 to 14399
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            14400 non-null  int64  
 1   DATE         14400 non-null  object 
 2   TIME         14400 non-null  object 
 3   PID          14400 non-null  int64  
 4   RSS          14400 non-null  int64  
 5   VSZ          14400 non-null  int64  
 6   MEM          14400 non-null  float64
 7   CPU          14400 non-null  float64
 8   CMD          14400 non-null  object 
 9   MAXMAPCOUNT  14400 non-null  int64  
 10  MAPS         0 non-null      float64
dtypes: float64(3), int64(5), object(3)
memory usage: 1.2+ MB
None


In [13]:
df2 = df.copy()
df2['datetime'] = (df['DATE'] + ' ' + df['TIME']).astype('datetime64')
df2 = df2.drop(['DATE', 'TIME', 'MAXMAPCOUNT', 'MAPS', 'N', 'CMD'], axis=1)
display(df2.head())
print(df2.info())

,PID,RSS,VSZ,MEM,CPU,datetime
0,563127,1331,4491,16.7,0.1,2024-02-16 00:00:02
1,3290,846,4490,10.6,0.1,2024-02-16 00:00:02
2,18023,529,1717,6.6,0.1,2024-02-16 00:00:02
3,675835,502,842,6.3,0.0,2024-02-16 00:00:02
4,675837,477,847,6.0,0.0,2024-02-16 00:00:02


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14400 entries, 0 to 14399
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   PID       14400 non-null  int64         
 1   RSS       14400 non-null  int64         
 2   VSZ       14400 non-null  int64         
 3   MEM       14400 non-null  float64       
 4   CPU       14400 non-null  float64       
 5   datetime  14400 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), int64(3)
memory usage: 675.1 KB
None


In [14]:
df2 = df2[df2['PID'] == 563127].set_index('datetime')
df2.resample('1H').mean()

,PID,RSS,VSZ,MEM,CPU
datetime,,,,,
2024-02-16 00:00:00,563127.0,1331.000000,4491.0,16.700000,0.1
2024-02-16 01:00:00,563127.0,1331.000000,4491.0,16.700000,0.1
2024-02-16 02:00:00,563127.0,1331.000000,4491.0,16.700000,0.1
2024-02-16 03:00:00,563127.0,1331.000000,4491.0,16.700000,0.1
2024-02-16 04:00:00,563127.0,1331.000000,4491.0,16.700000,0.1
2024-02-16 05:00:00,563127.0,1331.000000,4491.0,16.700000,0.1
2024-02-16 06:00:00,563127.0,1333.850000,4491.0,16.795000,0.1
2024-02-16 07:00:00,563127.0,1333.666667,4491.0,16.800000,0.1
2024-02-16 08:00:00,563127.0,1332.966667,4491.0,16.718333,0.1


In [20]:
df2.head()

,N,DATE,TIME,PID,RSS,VSZ,MEM,CPU,CMD,MAXMAPCOUNT,MAPS,datetime
0,1,2024-02-16,00:00:02,563127,1331,4491,16.7,0.1,java,65530,NaN,2024-02-16 00:00:02
1,2,2024-02-16,00:00:02,3290,846,4490,10.6,0.1,java,65530,NaN,2024-02-16 00:00:02
2,3,2024-02-16,00:00:02,18023,529,1717,6.6,0.1,java,65530,NaN,2024-02-16 00:00:02
3,4,2024-02-16,00:00:02,675835,502,842,6.3,0.0,/usr/local/bin/python,65530,NaN,2024-02-16 00:00:02
4,5,2024-02-16,00:00:02,675837,477,847,6.0,0.0,/usr/local/bin/python,65530,NaN,2024-02-16 00:00:02


In [21]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14400 entries, 0 to 14399
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   N            14400 non-null  int64         
 1   DATE         14400 non-null  object        
 2   TIME         14400 non-null  object        
 3   PID          14400 non-null  int64         
 4   RSS          14400 non-null  int64         
 5   VSZ          14400 non-null  int64         
 6   MEM          14400 non-null  float64       
 7   CPU          14400 non-null  float64       
 8   CMD          14400 non-null  object        
 9   MAXMAPCOUNT  14400 non-null  int64         
 10  MAPS         0 non-null      float64       
 11  datetime     14400 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(5), object(3)
memory usage: 1.3+ MB


In [ ]:
columns = ['N', 'DATE', 'TIME', 'RSS', '']